In [1]:
# Library
###############################################################################################################################
from pathlib import Path
data_dir = Path("/Users/mohammad/Desktop/Thesis/Thesis/SPARCS/Data")

import os
import numpy as np
import pandas as pd
from sklearn.base import clone

import statsmodels.api as sm
from scipy.stats import chi2

# Machine learning / preprocessing
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV
)


from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    auc,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# Missing data
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer

# Excel files
import openpyxl


In [ ]:
# Concat Data Suorce
#########################################################################################################################
diag = "acute hemorrhagic cerebrovascular disease"

#2019
hospdata_2019 = pd.read_csv(data_dir / "Hospital_Inpatient_Discharges__SPARCS_De-Identified___2019.csv")
hospdata_2019 = hospdata_2019[
    hospdata_2019["CCSR Diagnosis Description"]
    .str.strip()
    .str.lower()
    .eq(diag)
].copy()

#2020
hospdata_2020 = pd.read_csv(data_dir / "Hospital_Inpatient_Discharges__SPARCS_De-Identified___2020.csv")
hospdata_2020 = hospdata_2020[
    hospdata_2020["CCSR Diagnosis Description"]
    .str.strip()
    .str.lower()
    .eq(diag)
].copy()

#2021 
hospdata_2021 = pd.read_csv(data_dir / "Hospital_Inpatient_Discharges__SPARCS_De-Identified___2021.csv")
hospdata_2021 = hospdata_2021[
    hospdata_2021["CCSR Diagnosis Description"]
    .str.strip()
    .str.lower()
    .eq(diag)
].copy()

#2022 
hospdata_2022 = pd.read_csv(data_dir / "Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022.csv")
hospdata_2022 = hospdata_2022[
    hospdata_2022["CCSR Diagnosis Description"]
    .str.strip()
    .str.lower()
    .eq(diag)
].copy()

#2023
hospdata_2023 = pd.read_csv(data_dir / "Hospital_Inpatient_Discharges__SPARCS_De-Identified___2023.csv")
hospdata_2023 = hospdata_2023[
    hospdata_2023["CCSR Diagnosis Description"]
    .str.strip()
    .str.lower()
    .eq(diag)
].copy()


ny_hospdata = pd.concat(
    [hospdata_2019,hospdata_2020,hospdata_2021,hospdata_2022,hospdata_2023],ignore_index=True)

# Equivalent to !duplicated(...)
ny_hospdata = ny_hospdata.drop_duplicates().reset_index(drop=True)

ny_hospdata.to_csv(
    data_dir / "ny_hospdata.csv",
    index=False
)




In [3]:
# Suorce Data ############################################################################################################################
ny_hospdata = pd.read_csv(
    data_dir / "ny_hospdata.csv"
)

In [4]:
# Data Clean 
##################################################################################################################
################################################# Hospital Service Area #########################################################
ny_hospdata["Hospital Service Area"] = (
    ny_hospdata["Hospital Service Area"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)
ny_hospdata = ny_hospdata.dropna(subset=["Hospital Service Area"]).copy()
ny_hospdata["Hospital Service Area"] = ny_hospdata["Hospital Service Area"].astype("category")

################################################# Facility Name #########################################################
ny_hospdata["Facility Name"] = ny_hospdata["Facility Name"].astype("category")

################################################# Age Group #########################################################
# Remove patients age 0–17
ny_hospdata = ny_hospdata[
    ny_hospdata["Age Group"] != "0 to 17"
].copy()

# Define Age Group as a categorical variable
age_levels = [
    "18 to 29",
    "30 to 49",
    "50 to 69",
    "70 or Older"
]

ny_hospdata["Age Group"] = pd.Categorical(
    ny_hospdata["Age Group"],
    categories=age_levels,
    ordered=False
)

####################################################### Gender ###################################################
ny_hospdata = ny_hospdata.dropna(subset=["Gender"])
ny_hospdata = ny_hospdata[ny_hospdata['Gender'] != 'U']
ny_hospdata["Gender"] = ny_hospdata["Gender"].astype("category")

############################################################# Procedure ################################################
ny_hospdata["Procedure"] = (
    ny_hospdata["CCSR Procedure Description"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("Missing")
)
####################################################### APR Severity ####################################################
severity_levels = ["Minor", "Moderate", "Major", "Extreme"]

ny_hospdata["APR Severity of Illness Description"] = pd.Categorical(
    ny_hospdata["APR Severity of Illness Description"],
    categories= severity_levels,
    ordered = False
)

#################################################### APR Risk of Mortality #############################################
ny_hospdata["APR Risk of Mortality"] = pd.Categorical(
    ny_hospdata["APR Risk of Mortality"],
    categories= severity_levels,
    ordered=False
)

################################################# Type of Admission ################################################# 
# Remove unwanted admission types
ny_hospdata = ny_hospdata[
              (ny_hospdata["Type of Admission"]!='Newborn') &
              (ny_hospdata["Type of Admission"]!='Not Available')
            ].copy()

ny_hospdata["Type of Admission"] = ny_hospdata["Type of Admission"].astype("category") 

################################################# Dead ################################################# 
ny_hospdata["Dead"] = (
    ny_hospdata["Patient Disposition"] == "Expired"
).astype(int)

ny_hospdata["Dead"] = ny_hospdata["Dead"].map({
    1: "Dead",
    0: "Alive"
})

ny_hospdata["Dead"] = pd.Categorical(
    ny_hospdata["Dead"],
    categories=["Dead", "Alive"],
    ordered=False
)

############################################# APR Medical Surgical Description ##########################################
ny_hospdata["APR Medical Surgical Description"] = ny_hospdata["APR Medical Surgical Description"].astype("category")

############################################## Emergency Department Indicator ##########################################
ny_hospdata["Emergency Department Indicator"] = (
    ny_hospdata["Emergency Department Indicator"]
    .str.strip()
    .str.lower()
    .replace({
        "y": "Y",
        "n": "N",
        "true": "Y",
        "false": "N"
    })
)

ny_hospdata["Emergency Department Indicator"] = pd.Categorical(
   ny_hospdata["Emergency Department Indicator"],
   categories=["N","Y"],
   ordered= False
)

################################################# Length od Stay ################################################# 
ny_hospdata["Length of Stay"] = (
    ny_hospdata["Length of Stay"]
    .astype('string')
    .str.strip()
)
ny_hospdata["Length of Stay"] = ny_hospdata["Length of Stay"].replace(
    "120 +", "120"
)
ny_hospdata["Length of Stay"] = pd.to_numeric(
    ny_hospdata["Length of Stay"], errors= 'coerce'
)
ny_hospdata = ny_hospdata.dropna(subset=["Length of Stay"]).copy()

ny_hospdata["Length of Stay_Class"] = pd.cut(
    ny_hospdata["Length of Stay"],
    bins=[-np.inf, 2, 5, 10, 20, np.inf],
    labels=["0-2", "3-5", "6-10", "11-20", "21+"],
    right=True,
    ordered=False
)
ny_hospdata["Length of Stay_Class"] = (
    ny_hospdata['Length of Stay_Class']
    .astype('string')
    .str.strip()
)
ny_hospdata['Length of Stay']= (
    ny_hospdata["Length of Stay"]
    .astype('string')
    .str.strip()
)



In [5]:
# Selected Variables
###############################################################################################################################
selected_columns = [
    "Facility Name",
    "Discharge Year",
    "Hospital Service Area",
    "Age Group",
    "Gender",
    "Type of Admission",
    "Procedure",
    "APR Severity of Illness Description",
    "APR Risk of Mortality",
    "APR Medical Surgical Description",
    "Emergency Department Indicator",
    "Length of Stay_Class",
    "Dead"
]
model_df = ny_hospdata[selected_columns].copy()



In [6]:
# Temporal Split
###########################################################################################################################
train_data = model_df[
    model_df["Discharge Year"].isin([2019, 2020, 2021, 2022])
].copy()

test_data = model_df[
    model_df["Discharge Year"] == 2023
].copy()


In [7]:
# Group rare Procedure
minimum_procedure_count = 100

counts = train_data["Procedure"].value_counts()

common_procedures = counts[
    counts >= minimum_procedure_count
].index

def group_procedure(x):
    if x in common_procedures:
        return x
    else:
        return "Other"

train_data["Procedure"] = train_data["Procedure"].apply(group_procedure)
test_data["Procedure"] = test_data["Procedure"].apply(group_procedure)
  

In [8]:
# Model Formula
targert = "Dead"
predictors = [
    "Age Group",
    "Gender",
    "Type of Admission",
    "Procedure",
    "APR Severity of Illness Description",
    "APR Risk of Mortality",
    "APR Medical Surgical Description",
    "Emergency Department Indicator",
    "Length of Stay_Class"
]

X_train = train_data[predictors]
y_train = (train_data['Dead']=='Dead').astype(int)

X_test = test_data[predictors]
y_test = (test_data['Dead']=='Dead').astype(int)



In [ ]:
# Class distribution

train_distribution = (
    train_data["Dead"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Count")
)
train_distribution["Dataset"] = "Training_2019_2022"

test_distribution = (
    test_data["Dead"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Count")
)
test_distribution["Dataset"] = "Validation_2023"


class_distribution_glm = pd.concat(
    [train_distribution, test_distribution],
    ignore_index=True
)

class_distribution_glm["Proportion"] = (
    class_distribution_glm["Count"]
    / class_distribution_glm.groupby("Dataset")["Count"].transform("sum")
)

class_distribution_glm

In [ ]:
# GLM Model
glm_model = Pipeline([
    ("encode",OneHotEncoder(drop="first",handle_unknown="ignore")),
    ("model",LogisticRegression(penalty=None,max_iter=10000))
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=2026
)

cv_auc = cross_val_score(
    glm_model,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("Mean CV AUC:", cv_auc.mean())
glm_model.fit(X_train, y_train)
p_glm = glm_model.predict_proba(X_test)[:, 1]

Mean CV AUC: 0.9077233203670211


In [ ]:
# Core performance 

# ROC AUC
auc_glm = roc_auc_score(y_test, p_glm)

# PR AUC
precision, recall, _ = precision_recall_curve(y_test, p_glm)
auc_pr_glm = auc(recall, precision)

# Brier score
prevalence = y_test.mean()

brier_glm = np.mean((p_glm - y_test) ** 2)
brier_null = np.mean((prevalence - y_test) ** 2)

scaled_brier_glm = 1 - brier_glm / brier_null

# Obseryved / Expected deaths
observed_total = y_test.sum()
expected_total = p_glm.sum()
overall_oe = observed_total / expected_total

# O/E 95% CI
if observed_total == 0:
    overall_oe_lower = 0
else:
    overall_oe_lower = (
        chi2.ppf(0.025, 2 * observed_total)
        / (2 * expected_total)
    )

overall_oe_upper = (
    chi2.ppf(0.975, 2 * (observed_total + 1))
    / (2 * expected_total)
)

# ---------------- Calibration ----------------
p_clip = np.clip(p_glm, 1e-6, 1 - 1e-6)
logit_p = np.log(
    p_clip / (1 - p_clip)
)
# Calibration intercept and slope
X_cal = sm.add_constant(logit_p)

cal_model = sm.GLM(
    y_test,
    X_cal,
    family=sm.families.Binomial()
).fit()

calibration_intercept = cal_model.params[0]
calibration_slope = cal_model.params[1]

'''
cal_ci = cal_model.conf_int()
intercept_lower = cal_ci.iloc[0, 0]
intercept_upper = cal_ci.iloc[0, 1]
slope_lower = cal_ci.iloc[1, 0]
slope_upper = cal_ci.iloc[1, 1]
'''

# Calibration-in-the-large
citl_model = sm.GLM(
    y_test,
    np.ones((len(y_test), 1)),
    family=sm.families.Binomial(),
    offset=logit_p
).fit()

citl = citl_model.params[0]

core_metrics_glm = pd.DataFrame({
    "Metric": [
        "AUC_ROC",
        "AUC_PR_Dead",
        "Brier_Score",
        "Null_Brier",
        "Scaled_Brier",
        "Mortality_Prevalence",
        "Observed_Deaths",
        "Expected_Deaths",
        "Overall_OE",
        "Calibration_Intercept",
        "Calibration_Slope",
        "Calibration_in_the_Large"
    ],

    "Estimate": [
        auc_glm,
        auc_pr_glm,
        brier_glm,
        brier_null,
        scaled_brier_glm,
        prevalence,
        observed_total,
        expected_total,
        overall_oe,
        calibration_intercept,
        calibration_slope,
        citl
    ]
})

core_metrics_glm

In [ ]:
# Calibration deciles 
calibration_deciles_glm = pd.DataFrame({
    "Observed": y_test,
    "Predicted": p_glm
})

# Divide patients into 10 groups based on predicted mortality risk
calibration_deciles_glm["Risk_Decile"] = (
    calibration_deciles_glm["Predicted"]
    .rank(method="first", pct=True)
    .mul(10)
    .apply(np.ceil)
    .astype(int)
)

# Summarise calibration within each risk decile
calibration_deciles_glm = (
    calibration_deciles_glm
    .groupby("Risk_Decile", as_index=False)
    .agg(
        N=("Observed", "size"),
        Observed_Deaths=("Observed", "sum"),
        Expected_Deaths=("Predicted", "sum"),
        Mean_Predicted_Risk=("Predicted", "mean"),
        Observed_Mortality=("Observed", "mean")
    )
)

# Observed / Expected ratio within each decile
calibration_deciles_glm["O_E_Ratio"] = np.where(
    calibration_deciles_glm["Expected_Deaths"] > 0,
    calibration_deciles_glm["Observed_Deaths"]
    / calibration_deciles_glm["Expected_Deaths"],
    np.nan
)
calibration_deciles_glm



In [ ]:
# Threshold
###############################################################################################################################
# ---------------- Out-of-fold training predictions ----------------
# Equivalent to glm_model$pred in caret.
# Since RepeatedStratifiedKFold uses 5 folds x 3 repeats,
# each training observation receives 3 out-of-fold predictions.

oof_predictions = []

for train_idx, val_idx in cv.split(X_train, y_train):
    fold_model = clone(glm_model)
    fold_model.fit(
        X_train.iloc[train_idx],
        y_train.iloc[train_idx]
    )
    fold_probability = fold_model.predict_proba(
        X_train.iloc[val_idx]
    )[:, 1]
    fold_predictions = pd.DataFrame({
        "rowIndex": val_idx,
        "Actual": y_train.iloc[val_idx].to_numpy(),
        "Probability_Dead": fold_probability
    })
    oof_predictions.append(fold_predictions)

oof_predictions = pd.concat(
    oof_predictions,
    ignore_index=True
)

# Average repeated out-of-fold predictions for each patient
cv_predictions = (
    oof_predictions
    .groupby("rowIndex", as_index=False)
    .agg(
        Actual=("Actual", "first"),
        Probability_Dead=("Probability_Dead", "mean")
    )
)

# ---------------- Balanced accuracy function ----------------

  
def get_balanced_accuracy(probability, actual, threshold):

    try:

        predicted = (np.asarray(probability) >= threshold).astype(int)

        actual = np.asarray(actual).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            actual,
            predicted,
            labels=[0, 1]
        ).ravel()

        sensitivity = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else np.nan
        )

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        )

        if np.isnan(sensitivity) or np.isnan(specificity):
            return np.nan

        return (sensitivity + specificity) / 2

    except Exception:
        return np.nan


# ---------------- Search for optimal threshold ----------------

thresholds = np.round(
    np.arange(0.05, 0.61, 0.01),
    2
)

threshold_cv = pd.DataFrame({
    "Threshold": thresholds,
    "Balanced_Accuracy": [
        get_balanced_accuracy(
            cv_predictions["Probability_Dead"],
            cv_predictions["Actual"],
            threshold
        )
        for threshold in thresholds
    ]
})


# Select threshold with maximum balanced accuracy
if threshold_cv["Balanced_Accuracy"].isna().all():

    best_threshold_glm = 0.50

else:

    best_threshold_glm = threshold_cv.loc[
        threshold_cv["Balanced_Accuracy"].idxmax(),
        "Threshold"
    ]

print(
    "Best GLM threshold:",
    best_threshold_glm
)


# ---------------- Apply threshold to 2023 validation data ----------------

predicted_class_glm = pd.Categorical(
    np.where(
        p_glm >= best_threshold_glm,
        "Dead",
        "Alive"
    ),
    categories=["Dead", "Alive"]
)


# Binary version for performance calculations
predicted_binary_glm = (
    p_glm >= best_threshold_glm
).astype(int)


actual_binary_glm = (
    test_data["Dead"] == "Dead"
).astype(int).to_numpy()


# ---------------- Confusion matrix ----------------

try:

    cm_glm = confusion_matrix(
        actual_binary_glm,
        predicted_binary_glm,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm_glm.ravel()

except Exception:

    cm_glm = None
    tn = fp = fn = tp = np.nan


# ---------------- Classification metrics ----------------

if cm_glm is None:

    accuracy_glm = np.nan
    balanced_accuracy_glm = np.nan
    sensitivity_dead_glm = np.nan
    specificity_alive_glm = np.nan
    ppv_dead_glm = np.nan
    npv_alive_glm = np.nan

else:

    total = tn + fp + fn + tp

    # Accuracy
    accuracy_glm = (
        (tp + tn) / total
        if total > 0
        else np.nan
    )

    # Sensitivity / Recall for Dead
    sensitivity_dead_glm = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    # Specificity for Alive
    specificity_alive_glm = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    # Balanced Accuracy
    if (
        np.isnan(sensitivity_dead_glm)
        or np.isnan(specificity_alive_glm)
    ):
        balanced_accuracy_glm = np.nan
    else:
        balanced_accuracy_glm = (
            sensitivity_dead_glm
            + specificity_alive_glm
        ) / 2

    # Positive Predictive Value for Dead
    ppv_dead_glm = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    # Negative Predictive Value for Alive
    npv_alive_glm = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )


# ---------------- Final threshold performance table ----------------

threshold_2023_glm = pd.DataFrame({

    "Threshold": [
        best_threshold_glm
    ],

    "Accuracy": [
        accuracy_glm
    ],

    "Balanced_Accuracy": [
        balanced_accuracy_glm
    ],

    "Sensitivity_Dead": [
        sensitivity_dead_glm
    ],

    "Specificity_Alive": [
        specificity_alive_glm
    ],

    "PPV_Dead": [
        ppv_dead_glm
    ],

    "NPV_Alive": [
        npv_alive_glm
    ],

    "AUC_ROC": [
        auc_glm
    ],

    "AUC_PR_Dead": [
        auc_pr_glm
    ],

    "Brier_Score": [
        brier_glm
    ],

    "Scaled_Brier": [
        scaled_brier_glm
    ]
})


threshold_2023_glm

In [ ]:
# Death odds ratios
###############################################################################################################################
# Dead_binary = 1, therefore:
# OR > 1 = greater odds of death
# OR < 1 = lower odds of death

train_coef = train_data.copy()
train_coef["Dead_binary"] = (
    train_coef["Dead"] == "Dead"
).astype(int)

# Dense treatment-coded design matrix for statsmodels,
# using the same reference levels as the main model.
coef_encoder = make_one_hot_encoder(
    dense=True
)

X_coef = coef_encoder.fit_transform(
    train_coef[predictors]
)

coef_feature_names = (
    coef_encoder
    .get_feature_names_out(predictors)
    .tolist()
)

X_coef_sm = sm.add_constant(
    X_coef,
    has_constant="add"
)

glm_coef_model = sm.GLM(
    train_coef["Dead_binary"].to_numpy(),
    X_coef_sm,
    family=sm.families.Binomial()
).fit()

coef_terms = [
    "(Intercept)",
    *coef_feature_names
]

coefficient_table_glm = pd.DataFrame({
    "Term": coef_terms,
    "Estimate": np.asarray(glm_coef_model.params),
    "Std_Error": np.asarray(glm_coef_model.bse),
    "Z_Value": np.asarray(glm_coef_model.tvalues),
    "P_Value": np.asarray(glm_coef_model.pvalues)
})

coefficient_table_glm["Odds_Ratio_Death"] = np.exp(
    coefficient_table_glm["Estimate"]
)

coefficient_table_glm["OR_Lower_95CI"] = np.exp(
    coefficient_table_glm["Estimate"]
    - 1.96 * coefficient_table_glm["Std_Error"]
)

coefficient_table_glm["OR_Upper_95CI"] = np.exp(
    coefficient_table_glm["Estimate"]
    + 1.96 * coefficient_table_glm["Std_Error"]
)

coefficient_table_glm["Stability"] = np.select(
    [
        coefficient_table_glm["Term"].eq("(Intercept)"),
        (
            ~np.isfinite(
                coefficient_table_glm["Estimate"]
            )
            | ~np.isfinite(
                coefficient_table_glm["Std_Error"]
            )
        ),
        (
            coefficient_table_glm["Std_Error"] > 5
        )
        | (
            coefficient_table_glm["Estimate"].abs() > 20
        )
    ],
    [
        "Intercept",
        "Unstable - do not interpret",
        "Unstable - do not interpret"
    ],
    default="Stable"
)

# Remove non-finite values, matching the R code before Excel writing
coefficient_numeric_columns = [
    "Estimate",
    "Std_Error",
    "Z_Value",
    "P_Value",
    "Odds_Ratio_Death",
    "OR_Lower_95CI",
    "OR_Upper_95CI"
]

for col in coefficient_numeric_columns:
    coefficient_table_glm.loc[
        ~np.isfinite(
            coefficient_table_glm[col]
        ),
        col
    ] = np.nan

coefficient_table_glm


In [ ]:
# Reference levels and coefficient diagnostics
###############################################################################################################################

reference_levels_glm = pd.DataFrame({
    "Variable": predictors,
    "Reference_Level": [
        reference_levels[col]
        for col in predictors
    ]
})

# statsmodels GLMResults usually exposes `converged`
coef_converged = bool(
    getattr(
        glm_coef_model,
        "converged",
        True
    )
)

finite_se = (
    coefficient_table_glm["Std_Error"]
    .replace([np.inf, -np.inf], np.nan)
)

max_standard_error = (
    finite_se.max()
    if finite_se.notna().any()
    else np.nan
)

coefficient_diagnostics_glm = pd.DataFrame({
    "Item": [
        "Coefficient model converged",
        "Number of coefficients",
        "Number marked unstable",
        "Maximum standard error",
        "Minimum procedure count used"
    ],
    "Result": [
        coef_converged,
        len(coefficient_table_glm),
        int(
            (
                coefficient_table_glm["Stability"]
                == "Unstable - do not interpret"
            ).sum()
        ),
        max_standard_error,
        minimum_procedure_count
    ]
})

print("Reference levels:")
display(reference_levels_glm)

print("\nCoefficient diagnostics:")
display(coefficient_diagnostics_glm)


In [ ]:
# Patient predictions
###############################################################################################################################

patient_predictions_glm = pd.DataFrame({
    "Row_ID": np.arange(
        1,
        len(test_data) + 1
    ),
    "Facility_Name": (
        test_data["Facility Name"]
        .astype("string")
        .to_numpy()
    ),
    "Actual_Outcome": (
        test_data["Dead"]
        .astype("string")
        .to_numpy()
    ),
    "Dead_Binary": y_test,
    "Predicted_Probability_GLM": p_glm,
    "Predicted_Class_GLM": np.asarray(
        predicted_class_glm
    ).astype(str)
})

patient_predictions_glm


In [ ]:
# Plots
###############################################################################################################################

performance_colors = {
    "Higher than Expected": "#e74c3c",
    "As Expected": "#95a5a6",
    "Lower than Expected": "#2ecc71"
}

# ---------------- Plot 1: SMR by hospital ----------------
plot1_data = (
    hospital_smr_glm
    .sort_values("SMR_GLM")
    .reset_index(drop=True)
)

fig1, ax1 = plt.subplots(
    figsize=(14, 6)
)

x1 = np.arange(
    len(plot1_data)
)

for group_name, group_color in performance_colors.items():
    mask = (
        plot1_data["Performance_Descriptive"]
        == group_name
    )
    ax1.bar(
        x1[mask],
        plot1_data.loc[mask, "SMR_GLM"],
        width=0.7,
        label=group_name,
        color=group_color
    )

ax1.axhline(
    1,
    linestyle="--"
)

ax1.set_title(
    "Standardised Mortality Ratio (SMR) by Hospital\n"
    "GLM main-effects model | Reference line = 1"
)

ax1.set_xlabel("Hospital")
ax1.set_ylabel("SMR")

ax1.set_xticks(x1)
ax1.set_xticklabels(
    plot1_data["Facility Name"].astype(str),
    rotation=90,
    fontsize=7
)

ax1.legend(
    title="Performance",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=3
)

fig1.tight_layout()

# ---------------- Plot 2: observed vs expected ----------------
fig2, ax2 = plt.subplots(
    figsize=(9, 7)
)

# Scale point sizes while preserving N as the size variable
n_values = hospital_smr_glm["N"].astype(float)
if len(n_values) and n_values.max() > n_values.min():
    point_sizes = (
        30
        + 170
        * (
            (n_values - n_values.min())
            / (n_values.max() - n_values.min())
        )
    )
else:
    point_sizes = np.repeat(
        80.0,
        len(n_values)
    )

for group_name, group_color in performance_colors.items():
    mask = (
        hospital_smr_glm["Performance_Descriptive"]
        == group_name
    )

    ax2.scatter(
        hospital_smr_glm.loc[mask, "Expected_GLM"],
        hospital_smr_glm.loc[mask, "Observed"],
        s=np.asarray(point_sizes)[mask.to_numpy()],
        alpha=0.7,
        label=group_name,
        color=group_color
    )

max_oe = np.nanmax(
    np.concatenate([
        hospital_smr_glm["Expected_GLM"].to_numpy(dtype=float),
        hospital_smr_glm["Observed"].to_numpy(dtype=float)
    ])
) if len(hospital_smr_glm) else 1.0

ax2.plot(
    [0, max_oe],
    [0, max_oe],
    linestyle="--"
)

ax2.set_title(
    "Observed vs Expected Deaths by Hospital\n"
    "GLM main-effects model | Dashed line = O = E"
)

ax2.set_xlabel("Expected Deaths")
ax2.set_ylabel("Observed Deaths")
ax2.legend(
    title="Performance",
    loc="best"
)

fig2.tight_layout()

# ---------------- Plot 3: crude vs SMR rank ----------------
fig3, ax3 = plt.subplots(
    figsize=(8, 7)
)

for group_name, group_color in performance_colors.items():
    mask = (
        hospital_smr_glm["Performance_Descriptive"]
        == group_name
    )

    ax3.scatter(
        hospital_smr_glm.loc[mask, "Rank_Crude"],
        hospital_smr_glm.loc[mask, "Rank_SMR_GLM"],
        s=45,
        alpha=0.8,
        label=group_name,
        color=group_color
    )

rank_max = np.nanmax(
    np.concatenate([
        hospital_smr_glm["Rank_Crude"].to_numpy(dtype=float),
        hospital_smr_glm["Rank_SMR_GLM"].to_numpy(dtype=float)
    ])
) if len(hospital_smr_glm) else 1.0

ax3.plot(
    [0, rank_max],
    [0, rank_max],
    linestyle="--"
)

if spearman_result is None:
    spearman_text = (
        "Spearman not calculated"
    )
else:
    if spearman_p < 0.001:
        p_text = "< 0.001"
    else:
        p_text = f"= {spearman_p:.3g}"

    spearman_text = (
        f"Spearman rho = {spearman_rho:.3f} "
        f"(p {p_text})"
    )

if len(hospital_smr_glm):
    ax3.text(
        hospital_smr_glm["Rank_Crude"].max() * 0.18,
        hospital_smr_glm["Rank_SMR_GLM"].max() * 0.95,
        spearman_text
    )

ax3.set_title(
    "Hospital Rank: Crude vs Risk-Adjusted\n"
    "GLM main-effects model | Rank 1 = highest mortality measure"
)

ax3.set_xlabel(
    "Rank by Crude Mortality"
)

ax3.set_ylabel(
    "Rank by SMR"
)

ax3.legend(
    title="Performance",
    loc="best"
)

fig3.tight_layout()

# Save to temporary PNG files, matching the R tempfile()/ggsave() logic.
plot_files = [
    tempfile.NamedTemporaryFile(
        suffix=".png",
        delete=False
    ).name
    for _ in range(3)
]

plots = [
    fig1,
    fig2,
    fig3
]

plot_width = [
    14,
    9,
    8
]

plot_height = [
    6,
    7,
    7
]

plot_ok = []

for fig, path, width, height in zip(
    plots,
    plot_files,
    plot_width,
    plot_height
):
    try:
        fig.set_size_inches(
            width,
            height
        )
        fig.savefig(
            path,
            dpi=200,
            bbox_inches="tight"
        )
        plot_ok.append(True)
    except Exception:
        plot_ok.append(False)

print(
    "Plot files:",
    plot_files
)

print(
    "Saved successfully:",
    plot_ok
)

plt.show()
